In [95]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [96]:
model = pe.ConcreteModel()

Sets

In [97]:
model.orders = pe.Set(initialize = list(range(1,11)))
model.rolls = pe.Set(initialize = list(range(1,141)))

Parameters

In [98]:
unidades = [20,30,15,25,40,25,50,15,10,5]
dicc_unidades = dict(enumerate(unidades, start= 1))

model.units = pe.Param(model.orders, initialize=dicc_unidades)

longitudes = [0.5,0.75,1,1.25,1.5,1.75,2,2.25,2.5,2.75]
dicc_long = dict(enumerate(longitudes, start=1)) #muy iportante el dict, enumerate no devuelve un diccionario

model.lengths = pe.Param(model.orders, initialize=dicc_long)


Variables

In [99]:
model.n_parts = pe.Var(model.orders, model.rolls, within= pe.NonNegativeIntegers )
model.roll_used = pe.Var(model.rolls, within = pe.Binary)
model.waste = pe.Var(model.rolls, within= pe.NonNegativeReals)


Func Objetivo

In [100]:
def min_rolls_rule(model):
    return sum(model.roll_used[r] for r in model.rolls)

model.obj = pe.Objective(rule= min_rolls_rule, sense = pe.minimize)


Constraints

Max Cortes

In [101]:
def max_cuts_rule(model, r):
    return sum(model.n_parts[o,r] for o in model.orders) <= 4*model.roll_used[r]

model.max_cuts_constraint = pe.Constraint(model.rolls, rule = max_cuts_rule)

Suma = 3

In [102]:
def sum_3_rule(model, r):
    return sum(model.n_parts[o,r]* model.lengths[o] for o in model.orders ) + model.waste[r] == 3*model.roll_used[r]

model.sum_3_constraint = pe.Constraint(model.rolls, rule = sum_3_rule)

Cumplir demanda

In [103]:
def meet_demand_rule(model, o):
    return sum(model.n_parts[o,r] for r in model.rolls ) == model.units[o]

model.meet_demand_constraint = pe.Constraint(model.orders, rule = meet_demand_rule)

Si hay parte 1 -> parte 5 o parte 9

In [104]:
def P1_P5oP9_rule(model, r):
    return model.n_parts[1,r] <= 6*(model.n_parts[5,r]+model.n_parts[9,r])

model.P1_P9oP5_constraint = pe.Constraint(model.rolls, rule = P1_P5oP9_rule)

In [105]:

solver = po.SolverFactory("gurobi_direct")
results = solver.solve(model, tee=True)
print("Status:", results.solver.status)
print("Termination:", results.solver.termination_condition)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 430 rows, 1680 columns and 5040 nonzeros (Min)
Model fingerprint: 0x1dc9d458
Model has 140 linear objective coefficients
Variable types: 140 continuous, 1540 integer (140 binary)
Coefficient statistics:
  Matrix range     [5e-01, 6e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+00, 5e+01]

Presolve removed 0 rows and 140 columns
Presolve time: 0.01s
Presolved: 430 rows, 1540 columns, 4900 nonzeros
Variable types: 0 continuous, 1540 integer (840 binary)

Root relaxation: objective 1.187500e+02, 690 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap |

In [106]:
print(f"Rollos utilizados: {int(pe.value(model.obj))}")
print("=" * 50)

for r in model.rolls:
    if pe.value(model.roll_used[r]) > 0.5:
        
        print(f"\nROLLO {r}")
        print("-" * 30)
        
        longitud_usada = 0
        
        for o in model.orders:
            cantidad = round(pe.value(model.n_parts[o, r]))
            
            if cantidad > 0:
                longitud = pe.value(model.lengths[o])
                longitud_usada += cantidad * longitud
                
                print(
                    f"Pedido {o}: {cantidad} pieza(s) "
                    f"de {longitud:.2f} m"
                )
        
        waste = pe.value(model.waste[r])
        
        print(f"Longitud usada: {longitud_usada:.2f} m")
        print(f"Desperdicio:    {waste:.2f} m")

Rollos utilizados: 127

ROLLO 1
------------------------------
Pedido 3: 1 pieza(s) de 1.00 m
Pedido 7: 1 pieza(s) de 2.00 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

ROLLO 2
------------------------------
Pedido 2: 1 pieza(s) de 0.75 m
Pedido 8: 1 pieza(s) de 2.25 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

ROLLO 3
------------------------------
Pedido 5: 2 pieza(s) de 1.50 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

ROLLO 4
------------------------------
Pedido 1: 3 pieza(s) de 0.50 m
Pedido 5: 1 pieza(s) de 1.50 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

ROLLO 5
------------------------------
Pedido 2: 1 pieza(s) de 0.75 m
Pedido 8: 1 pieza(s) de 2.25 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

ROLLO 7
------------------------------
Pedido 4: 1 pieza(s) de 1.25 m
Pedido 6: 1 pieza(s) de 1.75 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

ROLLO 8
------------------------------
Pedido 5: 2 pieza(s) de 1.50 m
Longitud usada: 3.00 m
Desperdicio:    0.00 m

RO